# 事前準備：共通コードの実行
* このノートブックをコピーしたら，まずは以下の3つの共通コード（コードA～コードC）を順番に実行する
* これらの共通コードを実行しないと，それ以降のコードが実行できないので注意する
* また，コードA，コードB，及びコードC は，ノートブックに接続するたび毎回実行すること（ノートブックに接続中は，何度も実行する必要はない）
* 共通コードの詳細についての説明は割愛する（コードAとコードBの簡単な説明は[第2回のサンプルノートブック](https://colab.research.google.com/github/yoshida-nu/lecture_public/blob/main/datascience/doc/datascience_notebook02.ipynb)を参照）

In [ ]:
# コードA：日本語化ライブラリ導入
! pip install japanize-matplotlib | tail -n 1

In [ ]:
# コードB：共通事前処理

# B1:余分なワーニングを非表示にする
import warnings
warnings.filterwarnings('ignore')

# 必要ライブラリのimport
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import japanize_matplotlib # matplotlib日本語化対応
import seaborn as sns

# B2:データフレーム表示用関数
from IPython.display import display

# B3:表示オプション調整
np.set_printoptions(suppress = True, precision = 3) #numpyの浮動小数点の表示精度
pd.options.display.float_format = '{:.3f}'.format #pandasでの浮動小数点の表示精度
pd.set_option('display.max_columns', None) #データフレームですべての列データを表示

# B4:グラフのデフォルトフォント指定
plt.rcParams['font.size'] = 14

# 乱数の種
random_seed = 0

In [ ]:
# コードC：データのダウンロード
!curl -L -o diabetes.csv https://raw.githubusercontent.com/yoshida-nu/lecture_public/main/datascience/resources/11/diabetes.csv
!curl -L -o cinema.csv https://raw.githubusercontent.com/yoshida-nu/lecture_public/main/datascience/resources/11/cinema.csv
!curl -L -o employee_train.csv https://raw.githubusercontent.com/yoshida-nu/lecture_public/main/datascience/resources/11/employee_train.csv

# 目的と到達目標

## 目的
* 機械学習モデルの予測性能を評価するための代表的な指標を理解する
* K分割交差検証を理解する

## 到達目標
* 回帰モデルの代表的な評価指標である MAE，MSE，RMSE の意味を説明できる
* 分類モデルの混同行列，適合率，再現率，F1-score の意味を説明できる
* K分割交差検証の目的と手順を説明できる
* K分割交差検証を実践できる

# 予測性能評価

## なぜ予測性能を評価するのか
* 機械学習では，モデルを学習させるだけでは不十分である
* 学習済みモデルが，未知のデータに対してどの程度正しく予測できるかを確認する必要がある
* そのために，学習済みモデルによる予測結果と正解データを比較して評価指標を計算する
* 評価指標は，回帰と分類で異なる
  * 回帰：予測値と正解値の「ずれ」を評価
  * 分類：正しく分類できたか，どのように間違えたかを評価

## 今回扱う内容
* 回帰の予測性能評価
  * MAE（Mean Absolute Error: 平均絶対誤差）
  * MSE（Mean Squared Error: 平均2乗誤差）
  * RMSE（Root Mean Squared Error: 二乗平均平方根誤差）
* 分類の予測性能評価
  * 混同行列（Confusion Matrix）
  * 適合率（Precision）
  * 再現率（Recall / Sensitivity）
  * F1-score
* K分割交差検証（K-fold cross-validation）

# 回帰の予測性能評価

## 回帰における「誤差」
* 回帰では，正解値と予測値の差を使って予測性能を評価する
* 例：
  * 正解値が 10，予測値が 8 ⇒ 誤差の大きさは 2
  * 正解値が 10，予測値が 13 ⇒ 誤差の大きさは 3
* 正負の誤差をそのまま平均すると打ち消し合うため，絶対値や2乗を利用する

## MAE（平均絶対誤差）
* MAE（Mean Absolute Error）：予測誤差の絶対値を平均したもの
* 予測値と正解値が近いほど MAE は小さくなる
* MAE が 0 のとき，すべての予測値が正解値と一致している
* 元の目的変数と同じ単位で解釈できる
* MAEは，`sklearn.metrics` モジュールの `mean_absolute_error`関数で計算できる

**［以下のコードの処理内容］**
* 2行目: `sklearn.metrics` モジュールから `mean_absolute_error` 関数をインポートする
* 3行目: 正解値をリスト `y_true` に代入
* 4行目: 予測値をリスト `y_pred` に代入
* 5行目: `mean_absolute_error` 関数で，`y_true` と `y_pred` の MAE を計算し，変数 `mae` に代入
* 6行目: `print` 関数とf-stringを使って，MAE を小数点以下2桁まで表示

In [ ]:
# MAEの計算
from sklearn.metrics import mean_absolute_error
y_true = [10, 20, 30, 40]
y_pred = [12, 18, 33, 37]
mae = mean_absolute_error(y_true, y_pred)
print(f'MAE: {mae:.2f}')

## MSE（平均2乗誤差）
* MSE（Mean Squared Error）：予測誤差を2乗して平均したもの
* 誤差を2乗するため，大きな誤差の影響を強く受ける
* MSE が小さいほど，予測値と正解値が近い
* 誤差を2乗しているため，値の単位も2乗される
* MSEは，`sklearn.metrics` モジュールの `mean_squared_error`関数で計算できる

**［以下のコードの処理内容］**
* 2行目: `sklearn.metrics` モジュールから `mean_squared_error` 関数をインポートする
* 3行目: 正解値をリスト `y_true` に代入
* 4行目: 予測値をリスト `y_pred` に代入
* 5行目: `mean_squared_error` 関数で，`y_true` と `y_pred` の MSE を計算し，変数 `mse` に代入
* 6行目: `print` 関数とf-stringを使って，MSE を小数点以下2桁まで表示

In [ ]:
# MSEの計算
from sklearn.metrics import mean_squared_error
y_true = [10, 20, 30, 40]
y_pred = [12, 18, 33, 37]
mse = mean_squared_error(y_true, y_pred)
print(f'MSE: {mse:.2f}')

## RMSE（二乗平均平方根誤差）
* RMSE（Root Mean Squared Error）：MSE の平方根
* MSE と同様に，大きな誤差の影響を強く受ける
* 平方根をとるため，目的変数と同じ単位で解釈できる
* MAE と RMSE を比較すると，RMSE のほうが大きな誤差をより強く反映する

**［以下のコードの処理内容］**
* 2行目: 平方根を計算するために `math` モジュールの`sqrt`関数をインポートする
* 3行目: 正解値をリスト `y_true` に代入
* 4行目: 予測値をリスト `y_pred` に代入
* 5行目: `mean_squared_error` 関数で，`y_true` と `y_pred` の MSE を計算し，変数 `mse` に代入
* 6行目: `sqrt` 関数で MSE の平方根を計算し，変数 `rmse` に代入
* 7行目: `print` 関数とf-stringを使って，RMSE を小数点以下2桁まで表示

In [ ]:
# RMSEの計算
from math import sqrt
y_true = [10, 20, 30, 40]
y_pred = [12, 18, 33, 37]
mse = mean_squared_error(y_true, y_pred)
rmse = sqrt(mse)
print(f'RMSE: {rmse:.2f}')

## MAE・MSE・RMSEの比較
| 指標 | 定義 | 元の目的変数と同じ単位 | 大きな誤差を強く評価 |
|:--|:--|:--|:--|
| MAE | 誤差の絶対値の平均 | ✓ | - |
| MSE | 誤差の2乗の平均 | - | ✓ |
| RMSE | MSEの平方根 | ✓ | ✓ |

* どの指標も，小さいほど予測性能がよい
* 指標によって誤差の評価のしかたが異なるため，目的に応じて使い分ける

### 実習：MAE・MSE・RMSEの計算
* 以下の正解値と予測値に対して MAE，MSE，RMSE を計算する
  * 正解値: `[10, 20, 30, 40]`
  * 予測値: `[12, 18, 33, 71]`
* この例では，実際の値が「40」のときに「71」と予測しており，大きく外れている
* このような場合，MAE・MSE・RMSEにどのような影響が出るのかを確認する

**［実習内容］**
* 「以下のコードの処理内容」に従って，コードを完成させる
* 空行に適切なコードを記述する

**［以下のコードの処理内容］**
* 2～4行目: MAE，MSE，平方根を計算するために必要な関数をインポートする
* 5行目: 正解値を `y_true` に代入
* 6行目: 予測値を `y_pred` に代入
* 7行目: `y_true` と `y_pred` の MAE を計算し，変数 `mae` に代入
* 8行目: `y_true` と `y_pred` の MSE を計算し，変数 `mse` に代入
* 9行目: RMSE（MSEの平方根）を計算し，変数 `rmse` に代入
* 10～12行目: `print` 関数とf-stringを使って，MAE，MSE，RMSE を小数点以下2桁までそれぞれ表示

**［実行結果］**
```
MAE: 9.50
MSE: 244.50
RMSE: 15.64
```

In [ ]:
# MAE・MSE・RMSEの計算
from sklearn.metrics import mean_absolute_error
from sklearn.metrics import mean_squared_error
from math import sqrt
y_true = [10, 20, 30, 40]




print(f'MAE: {mae:.2f}')
print(f'MSE: {mse:.2f}')
print(f'RMSE: {rmse:.2f}')

# 分類の予測性能評価

## 分類では「間違い方」も重要
* 分類では，正しく分類できた割合だけを見ればよいとは限らない
* 特に，2カテゴリ分類では，どちらか一方を**対象カテゴリ**として考える
  * 対象カテゴリを「陽性」，それ以外のカテゴリを「陰性」と呼ぶこともある
* 例えば，客船の生存予測では，「生存」を対象カテゴリとすることができる
* このとき，次の2種類の間違いを区別することが重要である
  * 本当は対象カテゴリなのに，それ以外のカテゴリと予測する
  * 本当はそれ以外のカテゴリなのに，対象カテゴリと予測する
* どちらの間違いを重視するかは，分析の目的によって異なる

## 混同行列
* 混同行列：正解カテゴリと予測カテゴリの組み合わせごとの件数を表した表
* 2カテゴリ分類では，次の4種類に分けて考える
  * **True Positive（TP）**：対象カテゴリを対象カテゴリと正しく予測
  * **True Negative（TN）**：それ以外のカテゴリをそれ以外のカテゴリと正しく予測
  * **False Positive（FP）**：それ以外のカテゴリを対象カテゴリと誤って予測
  * **False Negative（FN）**：対象カテゴリをそれ以外のカテゴリと誤って予測
* 混同行列は，`sklearn.metrics` モジュールの `confusion_matrix` 関数で計算できる
  * 書式: `confusion_matrix([正解カテゴリのリスト], [予測カテゴリのリスト])`

**［以下のコードの処理内容］**
* 2行目: `sklearn.metrics` モジュールから `confusion_matrix` 関数をインポート
* 3行目: 正解カテゴリをリスト `t` に代入
  * `1` を対象カテゴリ，`0` をそれ以外のカテゴリとした（予測カテゴリも同様）
* 4行目: 予測カテゴリをリスト `pred` に代入
* 5行目: `confusion_matrix` 関数で混同行列を計算し，変数 `cm` に代入
* 6行目: `DataFrame` 関数で混同行列に行名・列名を付け，`display` 関数で表示（下表）

|  | 予測0 | 予測1 |
|---|---:|---:|
| 正解0 | **TN** | **FP** |
| 正解1 | **FN** | **TP** |

In [ ]:
# 混同行列の計算
from sklearn.metrics import confusion_matrix
t = [1, 1, 1, 1, 0, 0, 0, 0, 0, 0]
pred = [1, 1, 0, 0, 1, 0, 0, 0, 0, 0]
cm = confusion_matrix(t, pred)
display(pd.DataFrame(cm, index=['正解0', '正解1'], columns=['予測0', '予測1']))

## 適合率（Precision）
* 適合率：モデルが**対象カテゴリと予測したデータのうち，実際に対象カテゴリだった割合**
  * $\displaystyle{\mathrm{Precision}=\frac{TP}{TP+FP}}$
* 「対象カテゴリと予測した結果を，どの程度信用できるか」を表す
* 対象カテゴリではないデータを，対象カテゴリと誤って予測することを少なくしたい場合に重要となる
* つまり，**False Positive（FP）を少なくしたい場合**に重視される
* 適合率は，`sklearn.metrics` モジュールの `precision_score` 関数で計算できる
  * 書式: `precision_score([正解カテゴリのリスト], [予測カテゴリのリスト])`

**［以下のコードの処理内容］**
* 2行目: `sklearn.metrics` モジュールから `precision_score` 関数をインポート
* 3行目: 正解カテゴリをリスト `t` に代入
* 4行目: 予測カテゴリをリスト `pred` に代入
* 5行目: `precision_score` 関数で適合率を計算し，変数 `precision` に代入
* 6行目: `print` 関数とf-stringを使って，適合率（`precision`）を小数点以下2桁まで表示

In [ ]:
# 適合率の計算
from sklearn.metrics import precision_score
t = [1, 1, 1, 1, 0, 0, 0, 0, 0, 0]
pred = [1, 1, 0, 0, 1, 0, 0, 0, 0, 0]
precision = precision_score(t, pred)
print(f'適合率: {precision:.2f}')

## 再現率（Recall）
* 再現率：**実際に対象カテゴリであるデータのうち，正しく対象カテゴリと予測できた割合**
  * $\displaystyle{\mathrm{Recall}=\frac{TP}{TP+FN}}$
* 「対象カテゴリを，どの程度見逃さずに見つけられたか」を表す
* 対象カテゴリであるデータを，それ以外のカテゴリと誤って予測することを少なくしたい場合に重要となる
* つまり，**False Negative（FN）を少なくしたい場合**に重視される
* 適合率は，`sklearn.metrics` モジュールの `recall_score` 関数で計算できる
  * 書式: `recall_score([正解カテゴリのリスト], [予測カテゴリのリスト])`

**［以下のコードの処理内容］**
* 2行目: `sklearn.metrics` モジュールから `recall_score` 関数をインポート
* 3行目: 正解カテゴリをリスト `t` に代入
* 4行目: 予測カテゴリをリスト `pred` に代入
* 5行目: `recall_score` 関数で適合率を計算し，変数 `recall` に代入
* 6行目: `print` 関数とf-stringを使って，適合率（`recall`）を小数点以下2桁まで表示

In [ ]:
# 再現率の計算
from sklearn.metrics import recall_score
t = [1, 1, 1, 1, 0, 0, 0, 0, 0, 0]
pred = [1, 1, 0, 0, 1, 0, 0, 0, 0, 0]
recall = recall_score(t, pred)
print(f'再現率: {recall:.2f}')

## f1-score
* f1-score：適合率と再現率の両方を考慮した評価指標
  * $\displaystyle{\text{f1-score}=\frac{2\times \mathrm{Precision}\times \mathrm{Recall}}
{\mathrm{Precision}+\mathrm{Recall}}}$
* f1-scoreは，適合率と再現率の調和平均として計算される
* 調和平均とは，複数の値の平均を求める方法の一つで，一般的な平均値（算術平均）と比べて，**小さい値の影響を強く受ける** という特徴がある
* そのため，適合率と再現率のどちらか一方だけが高くても，F1-score は高くなりにくい
* 不均衡データの評価でもよく利用される
* f1-scoreは，`sklearn.metrics` モジュールの `f1_score` 関数で計算できる
  * 書式: `f1_score([正解カテゴリのリスト], [予測カテゴリのリスト])`

**［以下のコードの処理内容］**
* 2行目: `sklearn.metrics` モジュールから `f1_score` 関数をインポート
* 3行目: 正解カテゴリをリスト `t` に代入
* 4行目: 予測カテゴリをリスト `pred` に代入
* 5行目: `f1_score` 関数で f1-score を計算し，変数 `f1` に代入
* 6行目: `print` 関数とf-stringを使って，適合率（`f1`）を小数点以下2桁まで表示

In [ ]:
# f1-scoreの計算
from sklearn.metrics import f1_score
t = [1, 1, 1, 1, 0, 0, 0, 0, 0, 0]
pred = [1, 1, 0, 0, 1, 0, 0, 0, 0, 0]
f1 = f1_score(t, pred)
print(f'f1-score: {f1:.2f}')

## 実習：分類の評価指標を計算
* 以下の正解値と予測値に対して MAE，MSE，RMSE を計算する
  * 正解値: `[1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0]`
  * 予測値: `[1, 1, 1, 0, 0, 1, 0, 0, 0, 0, 0, 0]`

**［実習内容］**
* 「以下のコードの処理内容」に従って，コードを完成させる
* 空行に適切なコードを記述する

**［以下のコードの処理内容］**
* 2行目: 必要な関数（`precision_score`, `recall_score`, `f1_score`）をインポート
* 3行目: 正解カテゴリを `t2` に代入
* 4行目: 予測カテゴリを `pred2` に代入
* 5行目: `t2` と `pred2` の適合率を計算し，変数 `precision` に代入
* 6行目: `t2` と `pred2` の適合率を計算し，変数 `recall` に代入
* 7行目: `t2` と `pred2` の適合率を計算し，変数 `f1` に代入
* 8～10行目: `print` 関数とf-stringを使って，適合率，再現率，F1-score を小数点以下2桁までそれぞれ表示

**［実行結果］**
```
適合率: 0.75
再現率: 0.60
f1-score: 0.67
```

In [ ]:
# 分類の評価指標を計算
from sklearn.metrics import precision_score, recall_score, f1_score
t2 = [1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0]
pred2 = [1, 1, 1, 0, 0, 1, 0, 0, 0, 0, 0, 0]



print(f'適合率: {precision:.2f}')



## `classification_report`関数
* `sklearn.metrics` モジュールの `classification_report` 関数を使うと，分類モデルの性能指標（適合率，再現率，F1-score など）をまとめて確認できる
  * 書式: `classification_report([正解カテゴリのリスト], [予測カテゴリのリスト])`
* `classification_report` 関数の戻り値は文字列（`str`）
* 複数カテゴリ分類でも，各カテゴリについて評価指標を確認できる

### 利用例
**［以下のコードの処理内容］**
* 2行目: `sklearn.metrics` モジュールから `classification_report` 関数をインポート
* 3行目: 正解カテゴリをリスト `t` に代入
* 4行目: 予測カテゴリをリスト `pred` に代入
* 5行目: `classification_report` 関数で，分類モデルの性能指標まとめて計算し，変数 `report` に代入
* 6行目: `print` 関数を使って，結果（`report`）を表示

In [ ]:
# 評価指標をまとめて表示
from sklearn.metrics import classification_report
t = [1, 1, 1, 1, 0, 0, 0, 0, 0, 0]
pred = [1, 1, 0, 0, 1, 0, 0, 0, 0, 0]
report = classification_report(t, pred)
print(report)

### `classification_report`関数の出力の見方
* 主に次の指標が表示される
  * `precision`：適合率
  * `recall`：再現率
  * `f1-score`：適合率と再現率の調和平均
  * `support`：各カテゴリのデータ数
* 例えば，次のような結果（上のコードの実行結果）が表示されたとする

```text
              precision    recall  f1-score   support

           0       0.71      0.83      0.77         6
           1       0.67      0.50      0.57         4

    accuracy                           0.70        10
   macro avg       0.69      0.67      0.67        10
weighted avg       0.70      0.70      0.69        10
```

#### `0`，`1`の行
* 各カテゴリについて，それぞれの評価指標が表示される
  * `0` の行は カテゴリ0を対象カテゴリとして見たときの評価
  * `1` の行は カテゴリ1を対象カテゴリとして見たときの評価
* 上のコードで計算した値は，`1`の行に対応する
  * 適合率：0.67
  * 再現率：0.50
  * F1-score：0.57
  * 対象カテゴリのデータ数：4

#### `precision`
* モデルがそのカテゴリと予測したデータのうち，実際にそのカテゴリだった割合
* この例では，カテゴリ`1`の適合率は0.67である
* つまり，カテゴリ`1`と予測したデータの約67％が，実際にカテゴリ`1`であった

#### `recall`
* 実際にそのカテゴリであるデータのうち，正しくそのカテゴリと予測できた割合
* この例では，カテゴリ`1`の再現率は0.50である
* つまり，実際にカテゴリ`1`であるデータの50％を正しく見つけられた

#### `f1-score`
* 適合率と再現率の調和平均
* 適合率と再現率の両方を考慮した評価指標である
* この例では，カテゴリ`1`のF1-scoreは0.57である

#### `support`
* 各カテゴリに実際に含まれているデータ数
* この例では，
  * カテゴリ`0`：6件
  * カテゴリ`1`：4件
* `support`を見ることで，カテゴリごとのデータ数の偏りも確認できる

### 全体の評価（後半の行）

#### `accuracy`
* 全データのうち，正しく分類できたデータの割合
* この例では，正解率は0.70である
* つまり，10件中70％を正しく分類できた

#### `macro avg`
* 各カテゴリの評価指標を単純に平均した値
* 各カテゴリを同じ重みで評価する
* カテゴリごとのデータ数に偏りがある場合でも，少数カテゴリを同じ重みで評価できる

#### `weighted avg`
* 各カテゴリのデータ数（`support`）を考慮して平均した値
* データ数の多いカテゴリの影響が大きくなる
* カテゴリごとのデータ数に偏りがある場合，全体的な性能を確認する際に利用できる

### 出力を見るときのポイント
* まず，対象カテゴリの`precision`，`recall`，`f1-score`を確認する
* 次に，各カテゴリの`support`を確認し，データ数に偏りがないかを見る
* 不均衡データでは，正解率（`accuracy`）だけを見るとモデルの性能を誤解することがある
  * 例えば，多数派カテゴリだけを予測しても，全体の正解率が高くなる場合がある
  * そのため，適合率，再現率，F1-scoreなども確認することが重要である
* 必要に応じて，`macro avg`や`weighted avg`も確認する

# K分割交差検証

## ホールドアウト法の問題点
* これまでの講義では，データを訓練データとテストデータに1回だけ分割して評価してきた
* この方法をホールドアウト法と呼ぶ
* しかし，データの分け方によって評価結果が変わることがある
* 特にデータ数が少ない場合，1回だけの分割結果に評価が左右されやすい

## K分割交差検証の考え方
* K分割交差検証では，データを K 個のグループに分割する
* そのうち1つを検証用，残りを学習用として評価する
* 検証用に使うグループを入れ替えながら，同じモデルのもとで K回の学習と評価を行う
* 最後に K 回の評価結果を平均して，モデルの性能を確認する
* 1回だけの分割より，安定した評価を行いやすい

  <img src="./fig/illustration_Cross-Validation.jpg" width="700">

## 用いるデータ
* K分割交差検証の実践例として，csvファイル「diabetes.csv」を用いる
  * オリジナルのデータは，scikit-learnに付属している糖尿病（diabetes: ダイアベティス）の患者に関するデータ
  * ここで利用するデータは，オリジナルのデータを加工している
* 各列名と意味は下表のとおり
* 以降，このデータのことを「糖尿病データ」と呼ぶ
* 目的変数（正解データ）は数値なので，回帰問題として扱う
* データは適切に前処理されている（前処理不要）

| **列名** | **意味** |
|---|---|
| 年齢 | 患者の年齢 |
| 性別 | 患者の性別 |
| BMI | 肥満度を表す体格指数 |
| 血圧 | 患者の平均血圧 |
| 総コレステロール | 血液中の総コレステロール |
| LDL | 悪玉コレステロール |
| HDL | 善玉コレステロール |
| コレステロール比 | 総コレステロールとHDLの比 |
| 中性脂肪 | 血液中の中性脂肪に関連する指標 |
| 血糖 | 血液中の血糖値 |
| 進行度 | 1年後の糖尿病の進行度を表す指標（予測対象） |

**注意：** 「進行度」以外の列は，標準化後の数値であり，実際の年齢や血圧などの測定値ではない．

**［以下のコードの処理内容］**
* 2行目: csvファイルをDataFrameとして読み込んで，変数`df`に代入
* 3行目: `display`関数を使ってデータ（`df`の内容）を表示

In [ ]:
# 糖尿病データの読み込み
df = pd.read_csv('diabetes.csv')
display(df)

## `KFold`クラス
* `sklearn.model_selection`モジュールの `KFold` クラスを使うと，K分割交差検証におけるデータの分割方法を設定できる
* `KFold` クラスのオブジェクト自体が，K分割交差検証の設定内容に対応する
* このオブジェクトで，学習・評価を行うわけではない
* 学習・評価は，後述する `cross_validate` 関数を用いる
* 書式: `変数 = KFold(n_splits=[データ分割数 K]), shuffle=True, random_state=[乱数の種])`
  * `shuffle=True`でデータをランダムに分割する（`shuffle=False`にするとランダムに分割されない）

**［以下のコードの処理内容］**
* 2行目: `sklearn.model_selection`モジュールの `KFold` クラスをインポートする
* 3行目: `KFold` クラスのオブジェクトを生成し，変数`kf`に代入
  * `n_splits=3`：データを3分割
  * `shuffle=True`：分割前にデータをランダムに並べ替える
  * `random_state=random_seed`として，乱数を固定する ⇒ 結果が同じになる
* 4行目: `print`関数で「実行完了」を表示

In [ ]:
# K分割交差検証の分割方法を設定
from sklearn.model_selection import KFold
kf = KFold(n_splits=3, shuffle=True, random_state=random_seed)
print('実行完了')

## `cross_validate`関数
* `sklearn.model_selection`モジュールの `cross_validate` 関数を使うと，K分割交差検証による学習と評価をまとめて実行できる
* 書式: `cross_validate(estimator=[モデル（オブジェクト）], X=[説明変数], y=[目的変数], cv=[分割方法（KFoldクラスのオブジェクト）], scoring=[評価指標])`
* 引数 `scoring` でよく使われる評価指標を下表に示す
  * `cross_validate`関数の仕様で，評価指標は値が大きいほど，予測性能が高いと解釈できるように，平均2乗誤差と平均絶対誤差はマイナス1倍されている
* 戻り値は `dict` クラスのオブジェクトで，各回の評価結果などが含まれる

| **評価指標** | **書式** |
|---|---|
| 正解率 | `accuracy` |
| F値 | `f1_micro` |
| 適合率（の平均） | `precision_micro` |
| 再現率（の平均） | `recall_micro` |
| 決定係数 | `r2` |
| 平均2乗誤差のマイナス1倍 | `neg_mean_squared_error` |
| 平均絶対誤差のマイナス1倍 | `neg_absolute_error` |


**［以下のコードの処理内容］**
* 2行目: `sklearn.linear_model`モジュールの `LinearRegression` クラスをインポートする
* 3行目: `sklearn.model_selection`モジュールの `KFold` クラスと `cross_validate` クラスをインポートする
* 4行目: `KFold` クラスのオブジェクトを生成し，変数`kf`に代入
  * `n_splits=3`：データを3分割
  * `shuffle=True`：分割前にデータをランダムに並べ替える
  * `random_state=random_seed`として，乱数を固定する ⇒ 結果が同じになる
* 5行目: csvファイルをDataFrameとして読み込んで，変数`df`に代入
* 6行目: `df` から「進行度」列を除き，説明変数に対応する変数 `x` に代入
* 7行目: `df` の「進行度」列を取り出し，目的変数に対応する変数 `t` に代入
* 8行目: 線形回帰モデルの学習・評価をするためのオブジェクト（`LinearRegression` クラス）を作成し，変数 `model` に代入
* 9行目: 線形回帰モデルをK分割交差検証（K=3）で評価し，結果を `result` に代入
  * `estimator=model`：用いるモデル（`LinearRegression` クラスのオブジェクト `model`）を指定
  * `X=x`: 説明変数を指定
  * `y=t`: 目的変数を指定
  * `cv=kf`: 交差検証の分割方法（`KFold` クラスのオブジェクト `kf`）を指定
  * `scoring='r2'`: 評価指標として決定係数を指定
* 10行目: 交差検証の結果を DataFrame に変換し，表形式で表示する
  * 分割した各グループ（グループ 0 からグループ K-1 まで）の学習時間（fit_time），評価時間（score_time），決定係数（test_score）が表示される
* 11行目: 各分割で得られた決定係数 `result['test_score']` の平均値を計算し，変数 `r2_mean` に代入
  * K分割交差検証では，各分割で得られた評価結果（決定係数）の平均値をモデルの評価の目安として利用する
  * `cross_validate` の戻り値（`dict`）における，キー `test_score` の値に各回の評価値が配列として格納されている
* 12行目: 決定係数の平均値 `r2_mean` を小数点以下3桁まで表示

In [ ]:
# 線形回帰モデルを3分割交差検証で評価
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import KFold, cross_validate
kf = KFold(n_splits=3, shuffle=True, random_state=random_seed)
df = pd.read_csv('diabetes.csv')
x = df.drop(columns=['進行度'])
t = df['進行度']
model = LinearRegression()
result = cross_validate(estimator=model, X=x, y=t, cv=kf, scoring='r2')
display(pd.DataFrame(result))
r2_mean = result['test_score'].mean()
print(f'決定係数の平均値: {r2_mean:.3f}')

## 分類における交差検証
* 分類では，各カテゴリのデータ数の偏りにも注意する
* 分割後の各グループでカテゴリの割合が大きく変わると，評価結果が不安定になることがある
* そのため，分類ではカテゴリの割合をなるべく保ちながら分割する方法が利用される
* 具体的には，`sklearn.model_selection` モジュールの `StratifiedKFold` クラスを利用する
  * 書式: `StratifiedKFold(n_splits=[データ分割数 K]), shuffle=True, random_state=[乱数の種])`
  * 引数は `KFold` と同様

## 交差検証とチューニング
* モデルの設定値（ハイパーパラメータ）を変更しながら性能を比較することをチューニングと呼ぶ
* 同じテストデータを何度も見ながらチューニングすると，そのテストデータに合わせすぎる可能性がある
* そのため，交差検証を利用してチューニングを行い，最後に独立したテストデータで最終評価する考え方が重要である

# 回帰におけるK分割交差検証の実践例：映画データ

## 用いるデータと分析の目的
* 例として，csvファイル「cinema.csv」を用いる
* このデータは，これまでに公開された映画の興行収入のデータ（映画データと呼ぶ）である
  * 出典: 須藤秋良, 株式会社フレアリンク: スッキリわかるPythonによる機械学習入門, インプレス, 2020
  * 著者が作成したオリジナルのデータ
  * データ数は100個
* 映画データに対して，K分割交差検証を用いた線形回帰モデルの学習・評価を実践する

|**列名**| **内容** |
|:--|:--|
|cinema_id| 映画作品のID |
|SNS1| 公開後10日以内にSNS1で投稿された数 |
|SNS2| 公開後10日以内にSNS2で投稿された数 |
|actor| 主演俳優の昨年のメディア露出度（値が大きいほど露出している） |
|original| 原作の有無（ある「1」，ない「0」） |
|sales| 最終的な興行収入（単位: 万円） |

## 実習：映画データに対するK分割交差検証の実践
* K分割交差検証を用いた線形回帰モデルの学習・評価を行う
  * 説明変数: SNS1, SNS2, actor, original
  * 目的変数: sales
  * 乱数の種（`random_state`）は，すべて共通コードで定義している`random_seed`を用いる
  * 分割前にデータをランダムに並べ替える
  * 評価指標: 決定係数

**［実習内容］**
* 「以下のコードの処理内容」に従って，コードを完成させる
* 空行に適切なコードを記述する
  
**［以下のコードの処理内容］**
* 2行目: `sklearn.linear_model`モジュールの `LinearRegression` クラスをインポートする
* 3行目: `sklearn.model_selection`モジュールの `KFold` クラスと `cross_validate` クラスをインポートする
* 4行目: `KFold` クラスのオブジェクトを生成し，変数`kf`に代入
  * データを5分割
  * 分割前にデータをランダムに並べ替える
  * 乱数の種には `random_seed` を使う
* 5行目: 映画データのcsvファイルをDataFrameとして読み込んで，変数`df`に代入
* 6行目: `fillna`メソッドを使って，とし，欠損値を各列の平均値`df.mean()`にそれぞれ置き換えて変数`df`に代入
* 7行目: 説明変数の列名を要素とするリスト`['SNS1', 'SNS2', 'actor', 'original']`を変数`x_cols`に代入
* 8行目: 目的変数の列名のリスト`['sales']`を変数`t_col`に代入
* 9行目: DataFrame `df` から,`df[x_cols]`で，説明変数の列だけ取り出し変数`x`に代入
* 10行目: DataFrame `df` から,`df[t_col]`で目的変数の列だけ取り出し変数`t`に代入
* 11行目: 線形回帰モデルの学習・評価をするためのオブジェクト（`LinearRegression` クラス）を作成し，変数 `model` に代入
* 12行目: 線形回帰モデルをK分割交差検証（K=5）で評価し，結果を `result` に代入
  * 評価指標として決定係数を指定
* 13行目: 交差検証の結果を DataFrame に変換し，表形式で表示する
* 14行目: 各分割で得られた決定係数の平均値を計算し，変数 `r2_mean` に代入
* 15行目: 決定係数の平均値 `r2_mean` を小数点以下3桁まで表示

**［実行結果］**

  <img src="./fig/exercise_Cross-Validation_cinema.jpg" width="250">


In [ ]:
# 映画データに対するK分割交差検証の実践
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import KFold, cross_validate

df = pd.read_csv('cinema.csv')

x_cols = ['SNS1', 'SNS2', 'actor', 'original']


t = df[t_col]


display(pd.DataFrame(result))

print(f'決定係数の平均値: {r2_mean:.3f}')

# 分類におけるK分割交差検証の実践例：従業員データ

## 用いるデータと分析の目的
* 例として，csvファイル「employee_train.csv」を用いる
* このデータ（従業員データと呼ぶ）は，ある会社の従業員データベース一覧をイメージしている
  * 出典: 須出典:池田他: [実務で役立つPython機械学習入門 課題解決のためのデータ分析の基礎](https://www.shoeisha.co.jp/book/detail/9784798184890), 翔泳社, 2023
  * オリジナルのデータ: https://github.com/ml-pg-book/python-business-ml-starter
* 欠損値や外れ値はない
* 従業員データに対して，K分割交差検証を用いたロジスティック回帰モデルの学習・評価を実践する
* ロジスティック回帰モデルについては，第5回「[サンプルノートブック05](https://colab.research.google.com/github/yoshida-nu/lecture_public/blob/main/datascience/doc/datascience_notebook05.ipynb)」を参照
* 前述したように分類におけるK分割交差検証には，`sklearn.model_selection` モジュールの `StratifiedKFold` クラスを利用する

|**列名**| **意味** |
|:--|:--|
|leaving| 0: 離職してない / 1: 離職した |
|gender| 性別（女性 or 男性） |
|age| 年齢 |
|job_role| 職種 |
|job_satisfaction| 仕事に対する満足度（1～5の5段階評価） |
|environment_satisfaction| 職場環境に対する満足度（1～5の5段階評価） |
|over_time| 残業が多かったか（0: 少なかった / 1: 多かった） |
|income| 年収 |


## 実習：従業員データに対するK分割交差検証の実践
* K分割交差検証を用いたロジスティック回帰モデルの学習・評価を行う
  * 説明変数: age, job_satisfaction, environment_satisfaction, over_time, income
  * 目的変数: leaving 
  * 乱数の種（`random_state`）は，すべて共通コードで定義している`random_seed`を用いる
  * データを5分割する
  * 分割前にデータをランダムに並べ替える
  * 評価指標: 正解率

**［実習内容］**
* 「以下のコードの処理内容」に従って，コードを完成させる
* 空行に適切なコードを記述する
  
**［以下のコードの処理内容］**
* 2行目: `sklearn.linear_model`モジュールの `LogisticRegression` クラスをインポートする
* 3行目: `sklearn.model_selection`モジュールの `StratifiedKFold` クラスと `cross_validate` クラスをインポートする
* 4行目: `StratifiedKFold` クラスのオブジェクトを生成し，変数`skf`に代入
  * データを5分割
  * 分割前にデータをランダムに並べ替える
  * 乱数の種には `random_seed` を使う
* 5行目: 従業員データのcsvファイルを DataFrame として読み込んで，変数`df`に代入
* 6行目: 説明変数の列名を要素とするリスト`['age', 'job_satisfaction', 'environment_satisfaction', 'over_time', 'income']`を変数`x_cols`に代入
* 7行目: 目的変数の列名のリスト`['leaving ']`を変数`t_col`に代入
* 8行目: DataFrame `df` から,`df[x_cols]`で，説明変数の列だけ取り出し変数`x`に代入
* 9行目: DataFrame `df` から,`df[t_col]`で目的変数の列だけ取り出し変数`t`に代入
* 10行目: 線形回帰モデルの学習・評価をするためのオブジェクト（`LogisticRegression` クラス）を作成し，変数 `model` に代入
  * `random_state=random_seed`で，乱数を固定
* 11行目: 線形回帰モデルをK分割交差検証（K=5）で評価し，結果を `result` に代入
  * 評価指標として正解率を指定
* 12行目: 交差検証の結果を DataFrame に変換し，表形式で表示する
* 13行目: 各分割で得られた正解率の平均値を計算し，変数 `accuracy_mean` に代入
* 14行目: 正解率の平均値 `accuracy_mean` を小数点以下3桁まで表示

**［実行結果］**

  <img src="./fig/exercise_Cross-Validation_employee_train.jpg" width="250">


In [ ]:
# 従業員データに対するK分割交差検証の実践
from sklearn.linear_model import LogisticRegression


df = pd.read_csv('employee_train.csv')
x_cols = ['age', 'job_satisfaction', 'environment_satisfaction', 'over_time', 'income']


t = df[t_col]


display(pd.DataFrame(result))

print(f'正解率の平均値: {accuracy_mean:.3f}')

# 参考文献・資料
* 池田雄太郎, 田尻俊宗, 新保雄大: 実務で役立つPython機械学習入門 課題解決のためのデータ分析の基礎, 翔泳社, 2023
